# LayaWCAG Improved Training — Kaggle Notebook

Fine-tunes Laya (ModernBERT-large, frozen encoder) with a Linear(1024, 63) head
on WCAG 2.2 violation detection for higher precision.

**Improvements over baseline:**
- Focal Loss + class-balanced pos_weight for extreme label imbalance
- Gradient accumulation (effective batch 64)
- Cosine LR schedule with warmup
- Label smoothing for calibration
- Mixed precision (fp16) on T4
- Early stopping on val macro-F1
- Per-criterion metric reporting

Inputs: `/kaggle/input/<owner>/laya-wcag-data/` with `train_wcag_ibm.jsonl`
and `val_wcag_ibm.jsonl` (lines `{state, labels: [0|1 x 63}`).

In [ ]:
!pip install -q torch transformers datasets onnx onnxruntime onnxscript scikit-learn laya hf_transfer
print("=== STAGE 1: pip install complete ===")


In [ ]:
import os, json, glob, random, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from transformers import get_cosine_schedule_with_warmup
import laya
from datasets import Dataset as HFDataset
from collections import Counter
from pathlib import Path
from tqdm import tqdm
warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [ ]:
# ── WCAG 2.2 Criteria — EXACT canonical 63, matching the model + data labels ──
# These 63 labels are position-aligned with the 63-element label vector in the
# dataset. Do NOT add/reorder — labels[i] == WCAG_22_CRITERIA[i]['criterion'].
WCAG_22_CRITERIA = [
    {"criterion": "1.1.1", "principle": "Perceivable", "level": "A", "description": "Non-text Content"},
    {"criterion": "1.2.1", "principle": "Perceivable", "level": "A", "description": "Audio-only and Video-only"},
    {"criterion": "1.2.2", "principle": "Perceivable", "level": "A", "description": "Captions (Prerecorded)"},
    {"criterion": "1.2.3", "principle": "Perceivable", "level": "A", "description": "Audio Description or Media Alternative"},
    {"criterion": "1.2.4", "principle": "Perceivable", "level": "AA", "description": "Captions (Live)"},
    {"criterion": "1.2.5", "principle": "Perceivable", "level": "AA", "description": "Audio Description (Prerecorded)"},
    {"criterion": "1.3.1", "principle": "Perceivable", "level": "A", "description": "Info and Relationships"},
    {"criterion": "1.3.2", "principle": "Perceivable", "level": "A", "description": "Meaningful Sequence"},
    {"criterion": "1.3.3", "principle": "Perceivable", "level": "A", "description": "Sensory Characteristics"},
    {"criterion": "1.3.4", "principle": "Perceivable", "level": "AA", "description": "Orientation"},
    {"criterion": "1.3.5", "principle": "Perceivable", "level": "AA", "description": "Identify Input Purpose"},
    {"criterion": "1.3.6", "principle": "Perceivable", "level": "AAA", "description": "Identify Purpose"},
    {"criterion": "1.4.1", "principle": "Perceivable", "level": "A", "description": "Use of Color"},
    {"criterion": "1.4.2", "principle": "Perceivable", "level": "A", "description": "Audio Control"},
    {"criterion": "1.4.3", "principle": "Perceivable", "level": "AA", "description": "Contrast (Minimum)"},
    {"criterion": "1.4.4", "principle": "Perceivable", "level": "AA", "description": "Resize Text"},
    {"criterion": "1.4.5", "principle": "Perceivable", "level": "AA", "description": "Images of Text"},
    {"criterion": "1.4.6", "principle": "Perceivable", "level": "AA", "description": "Contrast (Enhanced)"},
    {"criterion": "1.4.7", "principle": "Perceivable", "level": "AA", "description": "Low or No Background Audio"},
    {"criterion": "1.4.8", "principle": "Perceivable", "level": "AAA", "description": "Visual Presentation"},
    {"criterion": "1.4.9", "principle": "Perceivable", "level": "AAA", "description": "Images of Text (No Exception)"},
    {"criterion": "1.4.10", "principle": "Perceivable", "level": "AA", "description": "Reflow"},
    {"criterion": "1.4.11", "principle": "Perceivable", "level": "AA", "description": "Non-text Contrast"},
    {"criterion": "1.4.12", "principle": "Perceivable", "level": "AA", "description": "Text Spacing"},
    {"criterion": "1.4.13", "principle": "Perceivable", "level": "AA", "description": "Content on Hover or Focus"},
    {"criterion": "2.1.1", "principle": "Operable", "level": "A", "description": "Keyboard"},
    {"criterion": "2.1.2", "principle": "Operable", "level": "A", "description": "No Keyboard Trap"},
    {"criterion": "2.1.4", "principle": "Operable", "level": "A", "description": "Character Key Shortcuts"},
    {"criterion": "2.2.1", "principle": "Operable", "level": "A", "description": "Timing Adjustable"},
    {"criterion": "2.2.2", "principle": "Operable", "level": "A", "description": "Pause, Stop, Hide"},
    {"criterion": "2.3.1", "principle": "Operable", "level": "A", "description": "Three Flashes or Below Threshold"},
    {"criterion": "2.4.1", "principle": "Operable", "level": "A", "description": "Bypass Blocks"},
    {"criterion": "2.4.2", "principle": "Operable", "level": "A", "description": "Page Titled"},
    {"criterion": "2.4.3", "principle": "Operable", "level": "A", "description": "Focus Order"},
    {"criterion": "2.4.4", "principle": "Operable", "level": "A", "description": "Link Purpose (In Context)"},
    {"criterion": "2.4.5", "principle": "Operable", "level": "AA", "description": "Multiple Ways"},
    {"criterion": "2.4.6", "principle": "Operable", "level": "AA", "description": "Headings and Labels"},
    {"criterion": "2.4.7", "principle": "Operable", "level": "AA", "description": "Focus Visible"},
    {"criterion": "2.4.11", "principle": "Operable", "level": "AA", "description": "Focus Not Obscured (Minimum)"},
    {"criterion": "2.4.12", "principle": "Operable", "level": "AAA", "description": "Focus Not Obscured (Enhanced)"},
    {"criterion": "2.4.13", "principle": "Operable", "level": "AAA", "description": "Focus Appearance"},
    {"criterion": "2.5.1", "principle": "Operable", "level": "A", "description": "Pointer Gestures"},
    {"criterion": "2.5.2", "principle": "Operable", "level": "A", "description": "Pointer Cancellation"},
    {"criterion": "2.5.3", "principle": "Operable", "level": "A", "description": "Label in Name"},
    {"criterion": "2.5.4", "principle": "Operable", "level": "A", "description": "Motion Actuation"},
    {"criterion": "2.5.5", "principle": "Operable", "level": "AAA", "description": "Target Size (Large)"},
    {"criterion": "2.5.6", "principle": "Operable", "level": "AAA", "description": "Concurrent Input Mechanisms"},
    {"criterion": "3.1.1", "principle": "Understandable", "level": "A", "description": "Language of Page"},
    {"criterion": "3.1.2", "principle": "Understandable", "level": "AA", "description": "Language of Parts"},
    {"criterion": "3.2.1", "principle": "Understandable", "level": "A", "description": "On Focus"},
    {"criterion": "3.2.2", "principle": "Understandable", "level": "A", "description": "On Input"},
    {"criterion": "3.2.3", "principle": "Understandable", "level": "AA", "description": "Consistent Navigation"},
    {"criterion": "3.2.4", "principle": "Understandable", "level": "AA", "description": "Consistent Identification"},
    {"criterion": "3.2.6", "principle": "Understandable", "level": "A", "description": "Consistent Help"},
    {"criterion": "3.3.1", "principle": "Understandable", "level": "A", "description": "Error Identification"},
    {"criterion": "3.3.2", "principle": "Understandable", "level": "A", "description": "Labels or Instructions"},
    {"criterion": "3.3.3", "principle": "Understandable", "level": "AA", "description": "Error Suggestion"},
    {"criterion": "3.3.4", "principle": "Understandable", "level": "AA", "description": "Error Prevention (Legal, Financial, Medical)"},
    {"criterion": "3.3.7", "principle": "Understandable", "level": "A", "description": "Redundant Input"},
    {"criterion": "3.3.8", "principle": "Understandable", "level": "AA", "description": "Accessible Authentication (Minimum)"},
    {"criterion": "4.1.1", "principle": "Robust", "level": "A", "description": "Parsing"},
    {"criterion": "4.1.2", "principle": "Robust", "level": "A", "description": "Name, Role, Value"},
    {"criterion": "4.1.3", "principle": "Robust", "level": "AA", "description": "Status Messages"},
]
num_criteria = len(WCAG_22_CRITERIA)
CRITERION_TO_IDX = {c["criterion"]: i for i, c in enumerate(WCAG_22_CRITERIA)}
assert num_criteria == 63, f'Expected 63 criteria, got {num_criteria}'
print(f"Defined {num_criteria} WCAG 2.2 criteria (canonical)")


In [ ]:
# ── Load base Laya model from the pre-staged Kaggle dataset (no runtime HF download) ──
import os, shutil, time as _t, glob

# Aggressive search: find laya.partaa anywhere under /kaggle/input
found = glob.glob("/kaggle/input/**/laya.partaa", recursive=True)
if not found:
    raise FileNotFoundError("laya.partaa not found anywhere under /kaggle/input. Contents: " + str(os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "NO /kaggle/input"))
STAGED = os.path.dirname(found[0])
print(f"=== STAGE 2: found dataset at {STAGED} ===")

BASE = "/kaggle/working/laya_base"
os.makedirs(os.path.join(BASE, "tokenizer"), exist_ok=True)
os.makedirs(os.path.join(BASE, "encoder"), exist_ok=True)
# Reassemble model.safetensors from the two parts
with open(os.path.join(BASE, "model.safetensors"), "wb") as out:
    for part in ["laya.partaa", "laya.partab"]:
        with open(os.path.join(STAGED, part), "rb") as f:
            shutil.copyfileobj(f, out)
# Place config / tokenizer / encoder files in the layout laya.load() expects
shutil.copy(os.path.join(STAGED, "config.json"),              os.path.join(BASE, "config.json"))
shutil.copy(os.path.join(STAGED, "rl_agent_config.json"),     os.path.join(BASE, "rl_agent_config.json"))
shutil.copy(os.path.join(STAGED, "enc_config.json"),          os.path.join(BASE, "encoder", "config.json"))
shutil.copy(os.path.join(STAGED, "tok_tokenizer.json"),       os.path.join(BASE, "tokenizer", "tokenizer.json"))
shutil.copy(os.path.join(STAGED, "tok_tokenizer_config.json"),os.path.join(BASE, "tokenizer", "tokenizer_config.json"))
print(f"=== STAGE 2: reassembled model.safetensors = {os.path.getsize(os.path.join(BASE,'model.safetensors')):,} bytes ===")
_t0 = _t.time()
import laya
agent = laya.load(BASE, device=device)
print(f"=== STAGE 2: local base model ready in {_t.time()-_t0:.1f}s (no HF download) ===")
base_model = agent.model
tokenizer = agent.tok
print(f"Model parameters: {sum(p.numel() for p in base_model.parameters()):,}")
print(f"Hidden size: {base_model.encoder.config.hidden_size}")


In [ ]:
# ── Model: frozen encoder + classification head ──
import torch.nn as nn

class LayaWCAGModel(nn.Module):
    def __init__(self, base_model, num_criteria=63, freeze_encoder=True):
        super().__init__()
        self.base = base_model
        self.num_criteria = num_criteria
        if freeze_encoder:
            for param in self.base.encoder.parameters():
                param.requires_grad = False
        self.classifier = nn.Linear(base_model.encoder.config.hidden_size, num_criteria)
        nn.init.xavier_uniform_(self.classifier.weight, gain=0.01)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.base.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True
            )
        cls_embedding = outputs.last_hidden_state[:, 0].float()
        logits = self.classifier(cls_embedding)
        return logits

model = LayaWCAGModel(base_model, num_criteria=num_criteria, freeze_encoder=True).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Model initialized; trainable: {trainable:,} / total: {total:,}")

In [ ]:
# ── Focal Loss for class imbalance ──
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25, pos_weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.pos_weight = pos_weight
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        bce_loss = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction="none"
        )
        pt = torch.exp(-bce_loss)
        focal_weight = (1 - pt) ** self.gamma
        if self.alpha >= 0:
            alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
            focal_weight = focal_weight * alpha_t
        return (focal_weight * bce_loss).mean()

In [ ]:
# ── Data loading (robust: walk /kaggle/input, print tree for debugging) ──
import os, json, glob

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

# Debug: what is actually mounted under /kaggle/input?
print("=== /kaggle/input tree ===")
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        for fn in sorted(files):
            print(f"  {os.path.join(root, fn)}")
else:
    print("  /kaggle/input does not exist")

def find_data_dir():
    # 1. Explicit known mount paths
    for c in ["/kaggle/input/hummern/laya-wcag-data", "/kaggle/input/laya-wcag-data"]:
        if os.path.isdir(c):
            return c
    # 2. Recursive walk for the training files
    if os.path.isdir("/kaggle/input"):
        for root, dirs, files in os.walk("/kaggle/input"):
            if "train_final.jsonl" in files or "train_wcag_ibm.jsonl" in files:
                return root
    return None

def find_train_file(d):
    for cand in ["train_wcag_ibm.jsonl", "train_final.jsonl"]:
        if os.path.isfile(os.path.join(d, cand)):
            return cand
    return None
def find_val_file(d):
    for cand in ["val_wcag_ibm.jsonl", "val_final.jsonl"]:
        if os.path.isfile(os.path.join(d, cand)):
            return cand
    return None

data_dir = find_data_dir()
if data_dir is None:
    raise FileNotFoundError(
        "Could not find training data under /kaggle/input. The dataset "
        "hummern/laya-wcag-data did not mount. Check it is attached and the "
        "kernel ran on the same owner."
    )
print(f"Data dir: {data_dir}")

tr_file = find_train_file(data_dir)
va_file = find_val_file(data_dir)
if tr_file is None:
    raise FileNotFoundError(f"No training file in {data_dir}")
train_rows = load_jsonl(os.path.join(data_dir, tr_file))
if va_file is not None:
    val_rows = load_jsonl(os.path.join(data_dir, va_file))
else:
    random.shuffle(train_rows)
    split_idx = int(len(train_rows) * 0.85)
    val_rows = train_rows[split_idx:]
    train_rows = train_rows[:split_idx]
print(f"Train: {len(train_rows)} ({tr_file}), Val: {len(val_rows)} ({va_file or 'split'})")

# Guard: label vectors must match the 63-criterion model (rejects old 73-criterion data)
if train_rows and len(train_rows[0]["labels"]) != num_criteria:
    raise ValueError(
        f"Dataset has label vectors of length {len(train_rows[0]['labels'])}, "
        f"but model expects {num_criteria}. Use hummern/laya-wcag-data-63."
    )
print(f"Label-length check OK: {num_criteria}-vectors")

# Compute pos_weight from training labels
labels_array = np.array([r["labels"] for r in train_rows])
pos_counts = labels_array.sum(axis=0)
neg_counts = len(train_rows) - pos_counts
pos_weight = torch.tensor([
    (neg_counts[i] / pos_counts[i]) if pos_counts[i] > 0 else 1.0
    for i in range(num_criteria)
], dtype=torch.float32).clamp(max=50.0)
print(f"pos_weight range: {pos_weight.min():.1f} .. {pos_weight.max():.1f}")
print(f"Criteria with positives: {(pos_counts > 0).sum()}/63")


In [ ]:
# ── Dataset & DataLoader ──
MAX_LEN = 256

class WCAGDataset(Dataset):
    def __init__(self, rows, tokenizer, max_len=256):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        enc = self.tokenizer(
            row["state"],
            truncation=True,
            max_length=self.max_len,
            padding=False,
            return_tensors=None,
        )
        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "labels": [float(v) for v in row["labels"]],
        }

def collate_fn(batch, pad_token_id=0):
    max_len = max(len(item["input_ids"]) for item in batch)
    b = len(batch)
    input_ids = torch.full((b, max_len), pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((b, max_len), dtype=torch.long)
    labels = torch.zeros((b, num_criteria), dtype=torch.float32)
    for i, item in enumerate(batch):
        sl = len(item["input_ids"])
        input_ids[i, :sl] = torch.tensor(item["input_ids"], dtype=torch.long)
        attention_mask[i, :sl] = torch.tensor(item["attention_mask"], dtype=torch.long)
        labels[i] = torch.tensor(item["labels"], dtype=torch.float32)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

train_dataset = WCAGDataset(train_rows, tokenizer, MAX_LEN)
val_dataset = WCAGDataset(val_rows, tokenizer, MAX_LEN)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id),
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id),
    pin_memory=True,
)

In [ ]:
# ── Training configuration ──
epochs = 20
batch_size = 16
grad_accum = 4
lr = 2e-4
weight_decay = 0.01
warmup_ratio = 0.1
max_grad_norm = 1.0
focal_gamma = 2.0
focal_alpha = 0.25
label_smoothing = 0.05
pos_weight_cap = 50.0
fp16 = True
patience = 5

pos_weight = pos_weight.to(device)
criterion = FocalLoss(
    gamma=focal_gamma,
    alpha=focal_alpha,
    pos_weight=pos_weight,
    label_smoothing=label_smoothing,
)
optimizer = torch.optim.AdamW(
    model.classifier.parameters(), lr=lr, weight_decay=weight_decay
)
num_training_steps = len(train_loader) * epochs // grad_accum
num_warmup_steps = int(num_training_steps * warmup_ratio)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=num_training_steps
)
scaler = GradScaler() if fp16 and device == "cuda" else None

print(f"Total training steps: {num_training_steps}")
print(f"Warmup steps: {num_warmup_steps}")
print(f"Gradient accumulation: {grad_accum} (effective batch {batch_size * grad_accum})")

In [ ]:
# ── Metrics ──
def compute_metrics(logits, labels, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    tp = (preds * labels).sum(0)
    fp = (preds * (1 - labels)).sum(0)
    fn = ((1 - preds) * labels).sum(0)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return {
        "macro_f1": f1.mean().item(),
        "macro_precision": precision.mean().item(),
        "macro_recall": recall.mean().item(),
        "micro_f1": (2 * tp.sum() / (tp.sum() + fp.sum() + tp.sum() + fn.sum() + 1e-8)).item(),
        "per_criterion_f1": f1.tolist(),
        "per_criterion_precision": precision.tolist(),
        "per_criterion_recall": recall.tolist(),
    }

def evaluate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            loss = criterion(logits, batch["labels"].to(device))
            total_loss += loss.item()
            all_logits.append(logits.cpu())
            all_labels.append(batch["labels"].cpu())
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    metrics = compute_metrics(all_logits, all_labels)
    metrics["loss"] = total_loss / len(val_loader)
    return metrics

In [ ]:
# ── Training loop ──
os.makedirs("/kaggle/working/wcag_model", exist_ok=True)
best_f1 = 0.0
bad_epochs = 0
best_state = None

print(f"=== STAGE 3: training start ({epochs} epochs, {len(train_rows)} train rows) ===")
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
    for step, batch in enumerate(pbar):
        logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        loss = criterion(logits, batch["labels"].to(device)) / grad_accum
        if scaler:
            scaler.scale(loss).backward()
        else:
            loss.backward()
        total_loss += loss.item() * grad_accum
        if (step + 1) % grad_accum == 0:
            if scaler:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        pbar.set_postfix({"loss": f"{loss.item() * grad_accum:.4f}"})
    train_loss = total_loss / len(train_loader)

    val_metrics = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}/{epochs} | Train: {train_loss:.4f} | Val: {val_metrics['loss']:.4f} | Macro F1: {val_metrics['macro_f1']:.4f} | Micro F1: {val_metrics['micro_f1']:.4f}")

    f1_scores = val_metrics["per_criterion_f1"]
    crit_names = [c["criterion"] for c in WCAG_22_CRITERIA]
    sorted_idx = np.argsort(f1_scores)
    print("  Worst:", [f"{crit_names[i]}:{f1_scores[i]:.3f}" for i in sorted_idx[:3]])
    print("  Best:  ", [f"{crit_names[i]}:{f1_scores[i]:.3f}" for i in sorted_idx[-3:]])

    if val_metrics["macro_f1"] > best_f1:
        best_f1 = val_metrics["macro_f1"]
        bad_epochs = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        torch.save(best_state, "/kaggle/working/wcag_model/best_model.pth")
        print(f"  -> New best macro F1: {best_f1:.4f}, saved")
    else:
        bad_epochs += 1
        print(f"  -> No improvement ({bad_epochs}/{patience})")
        if bad_epochs >= patience:
            print("Early stopping.")
            break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Loaded best model (macro F1: {best_f1:.4f})")

torch.save(model.state_dict(), "/kaggle/working/wcag_model/final_model.pth")
print("Done.")

In [ ]:
# ── ONNX export ──
import torch.onnx
import onnx
import onnxruntime as ort

model.eval()
dummy_ids = torch.ones(1, 64, dtype=torch.long).to(device)
dummy_mask = torch.ones(1, 64, dtype=torch.long).to(device)

torch.onnx.export(
    model,
    (dummy_ids, dummy_mask),
    "/kaggle/working/wcag_model/laya-wcag.onnx",
    opset_version=17,
    input_names=["input_ids", "attention_mask"],
    output_names=["criterion_scores"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "criterion_scores": {0: "batch_size"},
    },
    external_data=False,
)
print("ONNX export complete")

# Strip invalid num_outputs attribute from Split nodes (PyTorch 2.x bug)
m = onnx.load("/kaggle/working/wcag_model/laya-wcag.onnx")
n_split = 0
for node in m.graph.node:
    if node.op_type == "Split":
        keep = [a for a in node.attribute if a.name != "num_outputs"]
        if len(keep) != len(node.attribute):
            del node.attribute[:]
            node.attribute.extend(keep)
            n_split += 1
vi = m.graph.value_info
keep_vi = [v for v in vi if v.name not in {i.name for i in m.graph.initializer}]
del vi[:]
vi.extend(keep_vi)
onnx.save(m, "/kaggle/working/wcag_model/laya-wcag-clean.onnx")
print(f"Cleaned {n_split} Split nodes")
onnx.checker.check_model(onnx.load("/kaggle/working/wcag_model/laya-wcag-clean.onnx"))
print("ONNX check OK")

In [ ]:
# ── Quantize + externalize weights ──
from onnxruntime.quantization import quantize_dynamic, QuantType
import os, hashlib, json as json_lib

quantize_dynamic(
    "/kaggle/working/wcag_model/laya-wcag-clean.onnx",
    "/kaggle/working/wcag_model/laya-wcag-q8.onnx",
    weight_type=QuantType.QUInt8,
)
print(f"Quantized size: {os.path.getsize('/kaggle/working/wcag_model/laya-wcag-q8.onnx') / 1e6:.1f} MB")

# Externalize weights for browser streaming
if os.path.exists("/kaggle/working/wcag_model/laya-wcag-q8.onnx.data"):
    os.remove("/kaggle/working/wcag_model/laya-wcag-q8.onnx.data")
qm = onnx.load("/kaggle/working/wcag_model/laya-wcag-q8.onnx", load_external_data=False)
onnx.save_model(
    qm,
    "/kaggle/working/wcag_model/laya-wcag.onnx",
    save_as_external_data=True,
    all_tensors_to_one_file=True,
    size_threshold=1024,
    location="laya-wcag.onnx.data",
)

# Verify reload with ORT (files are already in /kaggle/working/wcag_model/, auto-packaged)
so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
sess = ort.InferenceSession(
    "/kaggle/working/wcag_model/laya-wcag.onnx",
    sess_options=so,
    providers=["CPUExecutionProvider"],
)
ids = np.array([[1, 100, 200, 300, 2]], dtype=np.int64)
att = np.ones_like(ids)
out = sess.run(None, {"input_ids": ids, "attention_mask": att})
print(f"ORT forward OK, output shape: {out[0].shape}")
print(f"Model packaged at /kaggle/working/wcag_model/")